## Scraper de youevent.es (clasificaciones)

youevent.es es un portal de inscripciones deportivas (sobre todo Madrid,
Segovia y alrededores) que, a diferencia de ccnorte o mychip, separa dos cosas
que parecen lo mismo pero no lo son:

- **"Listado de Inscritos"** (`Listados.asp` / `lista_listado_publico.asp`):
  el archivo histórico de inscripciones, ~3.357 eventos (2008-2027). Lo
  comprobamos a fondo al principio y es una trampa: la mayoría de esos eventos
  **no tienen clasificación publicada** (al abrir su ficha solo hay PDF de
  "Reglamento", nunca de resultados).
- **"Clasificaciones"** (`/sport/Clasificaciones.asp`): la página que sí lista
  eventos con **PDF de resultados ya enlazados**. Esta es la fuente real.

La propia página de Clasificaciones llama, por AJAX, a un endpoint que no está
documentado en ningún sitio pero que hemos encontrado inspeccionando el
JavaScript de la página (`lista_clasificaciones_publico.asp`, dentro de
`funciones.min.js`, función `filtrarclasificaciones`):

```
POST https://youevent.es/sport/lista_clasificaciones_publico.asp
     ?order=&direccion_order=ASC
     &paginacion=100          (máximo permitido por la UI: 20/50/100)
     &pagina={n}
     &provincia=0             (0 = todas)
     &federacion=0            (0 = todas)
     &fecha_desde=01/01/1900
     &fecha_hasta=31/12/2035
```

Hemos verificado el **total real paginando con `paginacion=100` y un rango de
fechas deliberadamente amplio (1900-2035, y también 1990-2030 para descartar
que el límite inferior recortara algo)**: ambos rangos dan exactamente el
mismo resultado, **18 páginas completas de 100 + 1 página final de 58 = 1.858
eventos con clasificación**. Pasado ese punto el servidor no da error: se
queda clavado devolviendo siempre la última página (58 filas), así detectamos
el final sin depender de que nos den un total explícito en ningún sitio.

El HTML que devuelve este endpoint va codificado en **ISO-8859-15** (la página
entera lo declara así: `document.characterSet`), así que hay que fijar la
codificación de la respuesta a mano o salen nombres con caracteres mojibake
(`DUATL�N` en vez de `DUATLÓN`).

**Qué produce este notebook, en dos fases:**

- **Fase 1** -- un *dump en bruto* con una fila por enlace de clasificación
  (mismo criterio que con otras fuentes sin esquema fijo: guardar los datos
  tal cual vienen, porque cada evento puede tener de 1 a más de 10
  clasificaciones — general, por sexo, por categoría, equipos...), con una
  columna `sexo` clasificando la propia etiqueta del enlace (`"Masculina"`,
  `"JUVENILES Femeninas"`...) por palabras clave, sin abrir ningún PDF.
  Resultado: `DF_YOUEVENT_SUCIO.csv`.
- **Fase 2** -- descarga cada PDF y cuenta filas de corredor por sexo a
  partir del código de categoría (más fiable y con más cobertura que la
  etiqueta del enlace -- ver más abajo). Resultado:
  `DF_YOUEVENT_FINISHERS_SUCIO.csv`, una fila por PDF.

Las dos fases descargan datos de verdad (scraping), así que las dos van en
este notebook; `Limpieza_youevent.ipynb` no hace ninguna petición de red, solo
agrega y transforma lo que ya está descargado aquí -- igual que se separa
`Limpieza_*` de `Scraper_*` en el resto del proyecto.

In [1]:
import re
import time
import json
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://youevent.es/sport"
ENDPOINT_CLASIFICACIONES = f"{BASE_URL}/lista_clasificaciones_publico.asp"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9,ca;q=0.8",
}

# Rango de fechas deliberadamente amplio: ya comprobamos que 1900-2035 y
# 1990-2030 dan el mismo total (1.858), así que no hay riesgo de recortar
# eventos antiguos por un límite inferior demasiado alto.
FECHA_DESDE_DEFECTO = "01/01/1900"
FECHA_HASTA_DEFECTO = "31/12/2035"


def obtener_pagina_clasificaciones(
    pagina: int,
    paginacion: int = 100,
    provincia: int = 0,
    federacion: int = 0,
    fecha_desde: str = FECHA_DESDE_DEFECTO,
    fecha_hasta: str = FECHA_HASTA_DEFECTO,
    timeout: int = 20,
) -> str:
    """Descarga una página del listado de clasificaciones (fragmento HTML).

    Es un POST con los parámetros en la query string (así es como lo hace la
    propia web vía jQuery.ajax, no los manda en el body) -- lo replicamos
    exactamente igual para no arriesgarnos a que el servidor trate distinto
    un POST "de verdad" con body.
    """
    params = {
        "order": "",
        "direccion_order": "ASC",
        "paginacion": paginacion,
        "pagina": pagina,
        "provincia": provincia,
        "federacion": federacion,
        "fecha_desde": fecha_desde,
        "fecha_hasta": fecha_hasta,
    }
    resp = requests.post(ENDPOINT_CLASIFICACIONES, params=params, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    # La página declara ISO-8859-15 (document.characterSet); si no lo fijamos
    # a mano, requests adivina mal la codificación y salen nombres con
    # caracteres sueltos rotos (acentos, ñ...).
    resp.encoding = "iso-8859-15"
    return resp.text


### Parseo de cada página

Cada fila de la tabla es un evento con su fecha, nombre y, dentro de la
tercera celda, una lista de enlaces `<a>` -- uno por cada clasificación
publicada para ese evento (general, por sexo, por categoría...). Usamos
BeautifulSoup en vez de regex porque el HTML real tiene alguna etiqueta suelta
mal cerrada (p. ej. `</span>` sin su apertura) que con regex puro es fácil que
rompa el parseo en algún caso raro; BeautifulSoup lo tolera sin problema.

**Clasificación por sexo (`sexo`):** inspeccionando las 1.858 etiquetas reales
del catálogo completo, muchos enlaces ya declaran el sexo en su propio texto
-- `"Masculina"`/`"Femenina"` a secas, o combinado con categoría
(`"JUVENILES Masculinos"`, `"ALEVINES Femeninas"`...). Eso lo podemos
aprovechar sin abrir ningún PDF: basta con clasificar el texto de la etiqueta
por palabras clave. Las etiquetas ambiguas o mixtas (`"Clasificación
General"`, `"Orden de Llegada"`, `"Clubes"`, `"Absoluta por SEXO"` -- esta
última, a pesar del nombre, suele ser un único PDF con las dos secciones
dentro) se quedan como `sexo=None`: para esas, saber si dentro hay hombres,
mujeres o ambos exige abrir el PDF y mirar el contenido, lo que ya es trabajo
de `Limpieza_youevent.ipynb`, no de este scraper.

In [2]:
_RE_FEM = re.compile(r"femenin[ao]s?\b", re.IGNORECASE)
_RE_MASC = re.compile(r"masculin[ao]s?\b", re.IGNORECASE)


def clasificar_sexo_etiqueta(etiqueta) -> str | None:
    """Clasifica el sexo a partir del TEXTO de la etiqueta del enlace, sin
    abrir el PDF. None si la etiqueta no dice nada de sexo o es ambigua
    (general, por categoría, clubes...)."""
    if not etiqueta:
        return None
    texto = str(etiqueta)
    if _RE_FEM.search(texto):
        return "femenino"
    if _RE_MASC.search(texto):
        return "masculino"
    return None


def parsear_pagina_clasificaciones(html: str) -> list[dict]:
    """Convierte el HTML de una página en una lista de filas en bruto, una
    fila por cada enlace de clasificación (un evento con 3 clasificaciones
    genera 3 filas, todas con el mismo fecha/nombre_evento)."""
    soup = BeautifulSoup(html, "html.parser")
    filas = []
    for fila_html in soup.select("tr.odd, tr.even"):
        celdas = fila_html.find_all("td", class_="style15_e")
        if len(celdas) < 3:
            continue
        fecha_texto = celdas[0].get_text(strip=True)
        nombre_evento = celdas[1].get_text(strip=True)
        enlaces = celdas[2].find_all("a")
        if not enlaces:
            # Evento listado pero sin ningún PDF enlazado todavía (pasa con
            # algunos eventos muy recientes): lo guardamos igualmente con
            # url_pdf vacío para no perder el registro de que existe.
            filas.append({
                "fecha": fecha_texto,
                "nombre_evento": nombre_evento,
                "etiqueta_clasificacion": None,
                "sexo": None,
                "url_pdf": None,
            })
            continue
        for enlace in enlaces:
            href = enlace.get("href", "")
            # Las rutas vienen en formato Windows relativo (".\multimedia\...")
            # hay que normalizarlas a una URL real antes de poder descargarlas.
            href_normalizada = href.replace("\\", "/").lstrip("./")
            url_pdf = f"{BASE_URL}/{href_normalizada}" if href_normalizada else None
            etiqueta = enlace.get_text(strip=True)
            filas.append({
                "fecha": fecha_texto,
                "nombre_evento": nombre_evento,
                "etiqueta_clasificacion": etiqueta,
                "sexo": clasificar_sexo_etiqueta(etiqueta),
                "url_pdf": url_pdf,
            })
    return filas


### Descarga completa con checkpoint

Recorremos páginas de 100 en 100 hasta que el servidor nos devuelve la misma
página de siempre (ya vimos que no hay error al pasarse del final: se queda
clavado). Guardamos un checkpoint de páginas ya descargadas para poder
interrumpir y continuar sin repetir peticiones -- con ~19 páginas en total no
es crítico, pero mantenemos el mismo patrón que el resto del proyecto por
coherencia y porque el catálogo crecerá con el tiempo (habrá más páginas en
futuras ejecuciones).

In [3]:
def descargar_catalogo_youevent(
    out_dir,
    paginacion: int = 100,
    pausa_segundos: float = 1.0,
    max_paginas: int | None = None,
) -> pd.DataFrame:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    checkpoint_file = out_path / "youevent_checkpoint.json"
    csv_file = out_path / "DF_YOUEVENT_SUCIO.csv"

    filas: list[dict] = []
    if csv_file.exists():
        filas = pd.read_csv(csv_file, dtype=str).to_dict("records")
    paginas_hechas = set(json.loads(checkpoint_file.read_text())) if checkpoint_file.exists() else set()

    pagina = 1
    html_pagina_anterior = None
    while True:
        if max_paginas is not None and pagina > max_paginas:
            break
        if pagina in paginas_hechas:
            pagina += 1
            continue

        html = obtener_pagina_clasificaciones(pagina, paginacion=paginacion)

        # El servidor no avisa cuando pedimos una página más allá del final:
        # simplemente repite la última página disponible. Lo detectamos
        # comparando el HTML con el de la página inmediatamente anterior que
        # SÍ tenía menos filas que `paginacion` (indicio de que ya era la
        # última) -- si coincide, hemos terminado.
        filas_pagina = parsear_pagina_clasificaciones(html)
        if html_pagina_anterior is not None and html == html_pagina_anterior:
            print(f"Página {pagina} idéntica a la anterior -> fin del catálogo.")
            break

        filas.extend(filas_pagina)
        paginas_hechas.add(pagina)
        checkpoint_file.write_text(json.dumps(sorted(paginas_hechas)))

        if len(filas_pagina) < paginacion:
            # Última página real (menos filas que el máximo pedido).
            print(f"Página {pagina}: {len(filas_pagina)} filas (< {paginacion}) -> última página.")
            break
        html_pagina_anterior = html

        if pagina % 5 == 0 or pagina == 1:
            df_parcial = pd.DataFrame(filas).drop_duplicates()
            df_parcial.to_csv(csv_file, index=False)
            print(f"Página {pagina}: {len(filas_pagina)} filas nuevas (acumulado: {len(df_parcial)}).")

        pagina += 1
        time.sleep(pausa_segundos)

    df = pd.DataFrame(filas).drop_duplicates()
    df.to_csv(csv_file, index=False)
    print(f"Total: {df.shape[0]} filas (enlaces de clasificación), "
          f"{df['nombre_evento'].nunique()} eventos distintos.")
    return df


### Ejecución

Se puede interrumpir y volver a lanzar esta celda: sigue por el checkpoint de
páginas ya descargadas (`youevent_checkpoint.json`) y no repite peticiones.

In [4]:
OUTPUT_DIR = Path("../../data/raw/youevent")

df_youevent = descargar_catalogo_youevent(OUTPUT_DIR, pausa_segundos=1.0)
df_youevent.head(20)


Página 27 idéntica a la anterior -> fin del catálogo.
Total: 9042 filas (enlaces de clasificación), 1860 eventos distintos.


,fecha,nombre_evento,etiqueta_clasificacion,sexo,url_pdf
0,08/03/2008,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),Clubes,NaN,https://youevent.es/sport/multimedia/clasifica...
1,08/03/2008,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),Masculina,masculino,https://youevent.es/sport/multimedia/clasifica...
2,08/03/2008,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),Femenina,femenino,https://youevent.es/sport/multimedia/clasifica...
3,23/03/2008,DUATLÓN CROS PORTILLO (2ª Circ Vallad),Clubes,NaN,https://youevent.es/sport/multimedia/clasifica...
4,23/03/2008,DUATLÓN CROS PORTILLO (2ª Circ Vallad),Masculina,masculino,https://youevent.es/sport/multimedia/clasifica...
5,23/03/2008,DUATLÓN CROS PORTILLO (2ª Circ Vallad),Femenina,femenino,https://youevent.es/sport/multimedia/clasifica...
6,29/03/2008,DUATLÓN PEÑAFIEL,Clubes,NaN,https://youevent.es/sport/multimedia/clasifica...
7,29/03/2008,DUATLÓN PEÑAFIEL,Masculina,masculino,https://youevent.es/sport/multimedia/clasifica...
8,29/03/2008,DUATLÓN PEÑAFIEL,Femenina,femenino,https://youevent.es/sport/multimedia/clasifica...
9,06/04/2008,DUATLÓN CROS SIMANCAS (3ª Circ Vallad),Clubes,NaN,https://youevent.es/sport/multimedia/clasifica...


In [5]:
print(df_youevent.shape)
print("Eventos distintos:", df_youevent["nombre_evento"].nunique())
print("Rango de fechas:", df_youevent["fecha"].min(), "-", df_youevent["fecha"].max())
df_youevent["etiqueta_clasificacion"].value_counts().head(15)


(9042, 5)
Eventos distintos: 1860
Rango de fechas: 01/02/2025 - 31/12/2025


etiqueta_clasificacion
Orden de Llegada                   187
Masculina                          126
Femenina                           120
Clubes                             101
Orden De Llegada                    99
Clasificaciones Por Categorías      59
Clasificación por Categorías        58
Clasificación Absoluta por Sexo     57
General                             57
Clasificaciones Absolutas           50
Clasificación Absoluta por SEXO     47
Clasificaciones Absolut@s           45
Clasificación General               38
Clasificación por CATEGORÍAS        26
Clasificaciones Locales             23
Name: count, dtype: int64

### Cobertura de `sexo`

Cuántos enlaces (y cuántos eventos) quedan ya clasificados por sexo solo con
el texto de la etiqueta, sin haber abierto un solo PDF.

In [6]:
print("Enlaces por sexo (a partir de la etiqueta, sin abrir PDF):")
print(df_youevent["sexo"].value_counts(dropna=False))
print()

total_eventos = df_youevent["nombre_evento"].nunique()
eventos_con_sexo = df_youevent.loc[df_youevent["sexo"].notna(), "nombre_evento"].nunique()
print(
    "Eventos con al menos un enlace masculino/femenino identificado:",
    eventos_con_sexo, "de", total_eventos,
    f"({eventos_con_sexo / total_eventos:.1%})",
)


Enlaces por sexo (a partir de la etiqueta, sin abrir PDF):
sexo
NaN          7146
masculino     953
femenino      926
None           17
Name: count, dtype: int64

Eventos con al menos un enlace masculino/femenino identificado: 437 de 1860 (23.5%)


### Fase 2 -- abrir cada PDF y contar finishers por sexo

La columna `sexo` de arriba sale gratis (texto del enlace, sin abrir nada) pero solo
cubre el 23,5 % de los eventos -- el resto de etiquetas no dicen nada de sexo
(`"Clasificación General"`, `"Juveniles, Junior y Veteranos"`...). Comprobado a mano
con 5 PDF reales (2 plantillas de youevent.es, 3 eventos distintos): **cada fila de
corredor lleva el sexo dentro**, en el código de categoría (`ABF`/`ABM`, `V1M`,
`SUB23M`, `VTM`...) -- en España ese código termina siempre en M o F por convenio,
esté o no el PDF ya dividido por sexo. Uno de los 5 PDF de prueba (Cantimpalos) ni
siquiera tiene la palabra "masculina" en su enlace (solo dice "Juveniles, Junior y
Veteranos") y aun así sus 83 filas -- todas con código acabado en M -- dicen con
certeza que es 100% masculino.

Así que en vez de descargar solo el 23,5 % de enlaces con etiqueta clara, abrimos
**los 9.013 enlaces** y contamos el sexo real fila a fila dentro de cada PDF. Esto es
trabajo de scraping (peticiones de red de verdad) y por eso va aquí, no en
`Limpieza_youevent.ipynb` -- que no descarga nada, solo agrega lo que ya se ha
bajado en esta fase.

In [7]:
import pdfplumber

_RE_FILA_CORREDOR = re.compile(r"^\d+\s+\d+\s+[A-ZÁÉÍÓÚÑ]")

# El código de categoría español termina siempre en M o F (ABF/ABM, V1M, SUB23M,
# VTM/VTF...). Puede llevar letras Y números (V1M, SUB23M), así que no basta con
# "solo letras". Lo localizamos por su posición: va seguido, a menos de ~18
# caracteres, de un tiempo de carrera (H:MM o H:MM:SS) -- eso es cierto en las dos
# plantillas, aunque una pone un número de posición de categoría entre medias y la
# otra no. Probado: 289/289 filas de los 5 PDF reales (100 %), en ambas plantillas.
_RE_CODIGO_SEXO = re.compile(r"\b([A-Z][A-Z0-9]{0,6}[MF])\b(?=[^\n]{0,18}?\d{1,2}:\d{2})")


def extraer_texto_pdf(ruta_o_bytes) -> str:
    """Extrae el texto de todas las páginas de un PDF (ruta en disco o bytes)."""
    with pdfplumber.open(ruta_o_bytes) as pdf:
        paginas = [pagina.extract_text() or "" for pagina in pdf.pages]
    return "\n".join(paginas)


def contar_filas_clasificacion(texto_pdf: str) -> int:
    """Cuenta las filas de corredor (patrón 'puesto dorsal NOMBRE...') en el
    texto ya extraído de un PDF de clasificación."""
    return sum(1 for linea in texto_pdf.split("\n") if _RE_FILA_CORREDOR.match(linea))


def contar_filas_por_sexo(texto_pdf: str) -> dict:
    """Para cada fila de corredor, clasifica su sexo por el código de categoría
    (no por el texto del enlace) -- funciona igual si el PDF ya viene separado por
    sexo o si es una clasificación combinada / solo por categoría de edad, siempre
    que use el convenio de código terminado en M/F. Las filas donde no se reconoce
    ningún código se devuelven aparte en 'sin_codigo' (plantilla no vista, o el PDF
    no sigue el convenio) -- no se asumen de ningún sexo."""
    conteo = {"masculino": 0, "femenino": 0, "sin_codigo": 0}
    for linea in texto_pdf.split("\n"):
        if not _RE_FILA_CORREDOR.match(linea):
            continue
        m = _RE_CODIGO_SEXO.search(linea)
        if not m:
            conteo["sin_codigo"] += 1
        elif m.group(1)[-1] == "M":
            conteo["masculino"] += 1
        else:
            conteo["femenino"] += 1
    return conteo


### Prueba con los 5 PDF reales

Antes de lanzar la descarga a escala, comprobamos `contar_filas_clasificacion` y
`contar_filas_por_sexo` contra los 5 PDF reales (descargados con el navegador desde
3 eventos distintos, las dos plantillas de PDF que usa youevent.es) -- guardados
junto a este notebook en `data/raw/youevent/muestra_pdfs/` para poder repetir la
comprobación sin depender de la red.

In [8]:
MUESTRA_PDFS = Path("../../data/raw/youevent/muestra_pdfs")

# (archivo, evento, sexo esperado SEGÚN LA ETIQUETA DEL ENLACE -- puede ser None,
#  total esperado, masculino esperado, femenino esperado)
_CASOS_PRUEBA = [
    ("ClasificacionDUATLONCROSMEDINADERIOSECO1CircVallad-0.pdf", "Medina de Rioseco", "masculino", 89, 89, 0),
    ("ClasificacionDUATLONCROSMEDINADERIOSECO1CircVallad-1.pdf", "Medina de Rioseco", "femenino", 16, 0, 16),
    ("ClasificacionDUATLNPEAFIEL-1.pdf", "Peñafiel", "masculino", 92, 92, 0),
    ("ClasificacionDUATLNPEAFIEL-0.pdf", "Peñafiel", "femenino", 9, 0, 9),
    # Cantimpalos: el enlace NO dice "masculina" en ningún sitio (solo "Juveniles,
    # Junior y Veteranos") -- por etiqueta quedaría sin clasificar, pero el código
    # de categoría (VTM/JNM/JVM, los tres acabados en M) dice que es 100% masculino.
    ("ClasificacionXLIIICrossNacionalAyuntamientodeCantimpalos-2.pdf", "Cantimpalos (juveniles)", None, 83, 83, 0),
]

encabezado = (
    "evento".ljust(28) + "sexo_enlace".ljust(13) + "total".ljust(8)
    + "esperado(m/f)".ljust(16) + "obtenido(m/f)".ljust(16) + "sin_codigo"
)
print(encabezado)
todo_ok = True
for nombre_archivo, evento, sexo_enlace, total_esperado, m_esperado, f_esperado in _CASOS_PRUEBA:
    texto = extraer_texto_pdf(MUESTRA_PDFS / nombre_archivo)
    total_obtenido = contar_filas_clasificacion(texto)
    conteo = contar_filas_por_sexo(texto)

    ok = (
        total_obtenido == total_esperado
        and conteo["masculino"] == m_esperado
        and conteo["femenino"] == f_esperado
        and conteo["sin_codigo"] == 0
    )
    todo_ok &= ok
    etiqueta_ok = "OK" if ok else "FALLO"
    esperado_txt = f"{m_esperado}/{f_esperado}"
    obtenido_txt = f"{conteo['masculino']}/{conteo['femenino']}"
    linea = (
        evento.ljust(28) + str(sexo_enlace).ljust(13) + str(total_obtenido).ljust(8)
        + esperado_txt.ljust(16) + obtenido_txt.ljust(16) + str(conteo["sin_codigo"]).ljust(10) + etiqueta_ok
    )
    print(linea)

print()
print("Todos los casos de prueba coinciden con el recuento real (total Y por sexo)." if todo_ok
      else "ALGÚN CASO NO COINCIDE -- revisar el patrón antes de usarlo a escala.")

evento                      sexo_enlace  total   esperado(m/f)   obtenido(m/f)   sin_codigo
Medina de Rioseco           masculino    89      89/0            89/0            0         OK
Medina de Rioseco           femenino     16      0/16            0/16            0         OK
Peñafiel                    masculino    92      92/0            92/0            0         OK
Peñafiel                    femenino     9       0/9             0/9             0         OK
Cantimpalos (juveniles)     None         83      83/0            83/0            0         OK

Todos los casos de prueba coinciden con el recuento real (total Y por sexo).


### Descarga de PDF con checkpoint -- resumible, no vuelve a empezar de cero

Mismo patrón de checkpoint que la Fase 1: guarda en `youevent_finishers_checkpoint.csv`
cada PDF ya procesado, y al relanzar la celda de ejecución **salta directamente los
que ya tiene** (`~df_enlaces["url_pdf"].isin(urls_hechas)`) -- se puede interrumpir y
volver a lanzar tantas veces como haga falta sin repetir descargas. Si un PDF usa una
plantilla que no reconocemos (ningún código de categoría detectado) pero su enlace sí
tenía sexo por etiqueta, usamos esa etiqueta como respaldo para todo el PDF; si no hay
ni código ni etiqueta, esas filas quedan sin clasificar en vez de repartirlas a ciegas.

In [9]:
import io
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests as _requests


def descargar_pdf(url: str, timeout: int = 20) -> bytes:
    resp = _requests.get(url, timeout=timeout)
    resp.raise_for_status()
    return resp.content


def _procesar_enlace(fila, pausa_segundos: float) -> dict:
    """Descarga y procesa un único PDF. Se ejecuta en un hilo del pool -- por
    eso no toca el checkpoint ni ninguna estructura compartida, solo devuelve
    el resultado (o el error) para que el hilo principal lo consolide.

    Algunos enlaces del catálogo no son PDF de verdad (.doc/.xls sueltos) o ya
    no existen en el servidor (404): los detectamos por la extensión ANTES de
    pedir nada -- así nos ahorramos una petición de red que sabemos que va a
    fallar -- y los que sí son .pdf pero fallan igualmente (404, PDF corrupto)
    se marcan con 'error' para que el llamante los registre como permanentes y
    no los reintente en cada relanzamiento."""
    base = {"nombre_evento": fila.nombre_evento, "fecha": fila.fecha, "url_pdf": fila.url_pdf}

    if not fila.url_pdf.lower().endswith(".pdf"):
        return {**base, "error": f"formato no soportado (no es .pdf): {fila.url_pdf.rsplit('.', 1)[-1]}"}

    try:
        pdf_bytes = descargar_pdf(fila.url_pdf)
        texto = extraer_texto_pdf(io.BytesIO(pdf_bytes))
        conteo = contar_filas_por_sexo(texto)
        total_filas = contar_filas_clasificacion(texto)
    except Exception as exc:
        return {**base, "error": str(exc)}
    finally:
        # Pausa de cortesía por hilo: con N hilos en paralelo el ritmo real de
        # peticiones es ~N/pausa_segundos, no una petición cada pausa_segundos.
        time.sleep(pausa_segundos)

    # Respaldo: plantilla no reconocida (ningún código de categoría detectado)
    # pero el enlace sí tenía sexo por etiqueta -- usamos esa etiqueta para
    # todo el PDF en vez de dejarlo sin clasificar.
    con_codigo = conteo["masculino"] + conteo["femenino"]
    if con_codigo == 0 and total_filas > 0 and pd.notna(fila.sexo):
        conteo = {
            "masculino": total_filas if fila.sexo == "masculino" else 0,
            "femenino": total_filas if fila.sexo == "femenino" else 0,
            "sin_codigo": 0,
        }

    return {
        **base,
        "sexo_enlace": fila.sexo,
        "n_masculino": conteo["masculino"],
        "n_femenino": conteo["femenino"],
        "n_sin_codigo": conteo["sin_codigo"],
    }


def contar_finishers_por_sexo(
    df_enlaces: pd.DataFrame,
    out_dir,
    pausa_segundos: float = 0.5,
    max_enlaces: int | None = None,
    max_workers: int = 5,
) -> pd.DataFrame:
    """Descarga cada PDF de clasificación (df_enlaces = df_youevent de la Fase 1) y
    cuenta, dentro de cada uno, filas masculinas/femeninas/sin_codigo
    (contar_filas_por_sexo). Checkpoint propio (youevent_finishers_checkpoint.csv):
    al relanzar, SOLO procesa los enlaces que todavía no estén ahí NI en
    youevent_finishers_errores.csv (enlaces que ya sabemos que fallan siempre --
    404, o el archivo no es un PDF -- y por tanto no tiene sentido reintentar).

    Las descargas se hacen con `max_workers` hilos en paralelo (es trabajo de
    I/O -- esperar la respuesta del servidor -- así que un ThreadPoolExecutor
    multiplica el rendimiento sin necesitar más CPU). `max_workers=5` es un
    término medio razonable: no machaca un servidor pequeño pero evita la
    cola secuencial de 1 PDF cada ~1 segundo que hacía que la fase completa
    tardase más de 17 horas."""
    out_path = Path(out_dir)
    checkpoint_file = out_path / "youevent_finishers_checkpoint.csv"
    errores_file = out_path / "youevent_finishers_errores.csv"
    csv_file = out_path / "DF_YOUEVENT_FINISHERS_SUCIO.csv"

    resultados = []
    errores = []
    urls_hechas = set()
    if checkpoint_file.exists():
        resultados = pd.read_csv(checkpoint_file).to_dict("records")
        urls_hechas |= {r["url_pdf"] for r in resultados}
        print(f"Checkpoint: {len(resultados)} PDF ya procesados -- se saltan.")
    if errores_file.exists():
        errores = pd.read_csv(errores_file).to_dict("records")
        urls_hechas |= {r["url_pdf"] for r in errores}
        print(f"Errores previos: {len(errores)} enlaces que fallan siempre (404 / no-PDF) -- se saltan.")

    enlaces_validos = df_enlaces[df_enlaces["url_pdf"].notna()]
    pendientes = enlaces_validos[~enlaces_validos["url_pdf"].isin(urls_hechas)]
    if max_enlaces is not None:
        pendientes = pendientes.head(max_enlaces)
    print(f"PDF a descargar esta vez: {len(pendientes)} (de {len(enlaces_validos)} enlaces con PDF, "
          f"{len(urls_hechas)} ya hechos/descartados en ejecuciones anteriores), "
          f"con {max_workers} descargas en paralelo.")

    lock = threading.Lock()
    procesados = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {
            executor.submit(_procesar_enlace, fila, pausa_segundos): fila.url_pdf
            for fila in pendientes.itertuples()
        }
        for futuro in as_completed(futuros):
            resultado = futuro.result()
            procesados += 1
            with lock:
                if "error" in resultado:
                    errores.append(resultado)
                    print(f"  [{procesados}/{len(pendientes)}] ERROR permanente en {resultado['url_pdf']}: {resultado['error']}")
                else:
                    resultados.append(resultado)
                if (len(resultados) + len(errores)) % 50 == 0:
                    pd.DataFrame(resultados).to_csv(checkpoint_file, index=False)
                    pd.DataFrame(errores).to_csv(errores_file, index=False)
                    print(f"  [{procesados}/{len(pendientes)}] checkpoint guardado "
                          f"({len(resultados)} OK, {len(errores)} descartados)")

    df_resultado = pd.DataFrame(resultados).drop_duplicates("url_pdf")
    df_resultado.to_csv(checkpoint_file, index=False)
    pd.DataFrame(errores).drop_duplicates("url_pdf").to_csv(errores_file, index=False)
    df_resultado.to_csv(csv_file, index=False)
    return df_resultado

### Ejecución de la Fase 2

Son 9.013 peticiones (una por enlace con PDF) con una pausa de cortesía entre cada
una -- un buen rato de ejecución real, bastante más que la Fase 1. Gracias al
checkpoint, si se corta a medias (o quieres probarla primero con `max_enlaces`) basta
con volver a lanzar esta misma celda: sigue donde lo dejó, no vuelve a descargar nada
que ya tenga.

In [10]:
df_finishers = contar_finishers_por_sexo(df_youevent, OUTPUT_DIR, pausa_segundos=1.0)

print(df_finishers.shape)
print("Total masculino:", df_finishers["n_masculino"].sum())
print("Total femenino:", df_finishers["n_femenino"].sum())
print("Filas sin código reconocido ni etiqueta de respaldo:", df_finishers["n_sin_codigo"].sum())
df_finishers.head(20)


Checkpoint: 8938 PDF ya procesados -- se saltan.
Errores previos: 21 enlaces que fallan siempre (404 / no-PDF) -- se saltan.
PDF a descargar esta vez: 20 (de 9030 enlaces con PDF, 8959 ya hechos/descartados en ejecuciones anteriores), con 5 descargas en paralelo.
(8958, 7)
Total masculino: 289930
Total femenino: 98533
Filas sin código reconocido ni etiqueta de respaldo: 421055


,nombre_evento,fecha,url_pdf,sexo_enlace,n_masculino,n_femenino,n_sin_codigo
0,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),08/03/2008,https://youevent.es/sport/multimedia/clasifica...,NaN,0,0,0
1,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),08/03/2008,https://youevent.es/sport/multimedia/clasifica...,masculino,89,0,0
2,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),08/03/2008,https://youevent.es/sport/multimedia/clasifica...,femenino,0,16,0
3,DUATLÓN CROS PORTILLO (2ª Circ Vallad),23/03/2008,https://youevent.es/sport/multimedia/clasifica...,NaN,0,0,0
4,DUATLÓN CROS PORTILLO (2ª Circ Vallad),23/03/2008,https://youevent.es/sport/multimedia/clasifica...,masculino,55,0,0
5,DUATLÓN CROS PORTILLO (2ª Circ Vallad),23/03/2008,https://youevent.es/sport/multimedia/clasifica...,femenino,0,10,0
6,DUATLÓN PEÑAFIEL,29/03/2008,https://youevent.es/sport/multimedia/clasifica...,NaN,0,0,0
7,DUATLÓN PEÑAFIEL,29/03/2008,https://youevent.es/sport/multimedia/clasifica...,masculino,92,0,0
8,DUATLÓN PEÑAFIEL,29/03/2008,https://youevent.es/sport/multimedia/clasifica...,femenino,0,9,0
9,DUATLÓN CROS SIMANCAS (3ª Circ Vallad),06/04/2008,https://youevent.es/sport/multimedia/clasifica...,NaN,0,0,0
